In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
import seaborn as sns
import matplotlib.pyplot as plt
tickers = ['^GSPC','AAPL', 'MSFT', "GOOGL"]
print("Fetching historical data...")
raw_data=yf.download(tickers,period="5y")['Close']
print("Handling missing values...")
cleaned_data=raw_data.ffill().bfill()
print("Calculating log returns...")
log_returns = np.log(cleaned_data / cleaned_data.shift(1)).dropna()
print("Generating covariance matrix...")
cov_matrix = log_returns.cov()
print("Generating correlation matrix...")
corr_matrix = log_returns.corr()
output_dir=os.path.join("data","processed")
os.makedirs(output_dir, exist_ok=True)
log_returns.to_csv(os.path.join(output_dir,"processed_log_returns.csv"))
print(f"Processed datasetsaved to {output_dir}")
plt.figure(figsize=(10, 5))
for ticker in tickers:
    sns.kdeplot(log_returns[ticker], label=ticker,fill=True, alpha=0.2)
plt.title("Daily Log Return Distributions")
plt.xlabel("Log Return")
plt.legend()
plt.show()

: 

In [ ]:
import os
import numpy as np
import pandas as pd
from pipeline import DataIngestionPipeline
BASE_DIR="data"
PROCESSED_DIR=os.path.join(BASE_DIR,"processed")
os.makedirs(PROCESSED_DIR,exist_ok=True)
output_path=os.path.join(PROCESSED_DIR,"processed_log_returns.csv")
pipeline=DataIngestionPipeline(base_dir=BASE_DIR)
etfs_df=pipeline.aggregate_category("etfs")
stocks_df=pipeline.aggregate_category("stocks")
macro_df=pipeline.aggregate_category("macro")
price_matrices=[df for df in [etfs_df,stocks_df,macro_df]if not df.empty]
master_price_df=pd.concat(price_matrices,axis=1)
log_returns_df=np.log(master_price_df)-np.log(master_price_df.shift(1))
log_returns_df=log_returns_df.dropna(how="all").ffill().bfill()
log_returns_df.to_csv(output_path)
print(f"Matrix saved directly to :{output_path}")
log_returns_df.head()